# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 4: CTR / Engagement Opportunity Scoring.**

I'm picking this lane because it's the narrowest, most falsifiable question of the four: for pages that are
already **visible** (they rank, they get impressions), is the click-through rate weak *relative to other
pages at the same position*? Position dominates CTR — a page-1 page and a "deep" page are not directly
comparable — so the interesting question isn't "which pages have low CTR" but "which pages under-perform
**their own tier's** expectation." That's a well-defined, checkable comparison, not a vague "find bad pages"
task, and it points straight at a concrete editorial action (rewrite title/meta, improve snippet structure)
rather than a vague "refresh everything" pile. Section 3 below shows the position-tier CTR spread and the
size of the review-candidate pool on the starter data, which is why I think this lane is worth the next
7 weeks rather than a dead end.

I may confirm or swap this by end of Week 4 once I've looked at the warehouse release, per the guide.


In [3]:
# Setup — load the starter dataset once, reuse it in the cells below.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"rows: {len(df):,}  |  columns: {df.shape[1]}  |  clients: {df.client_id.nunique()}")


rows: 30,000  |  columns: 44  |  clients: 32


## 2. The question: decision, action, cost of a wrong call

**Unit of analysis:** one content page (`content_id`), summarized over its trailing 90-day window.

**The question:** among pages that already have real search visibility (they get meaningful impressions
at a real average position), which ones are capturing fewer clicks than other pages sitting at the *same*
position tier — and are therefore good candidates for a title/meta/snippet review?

**The decision it improves:** which page an SEO reviewer opens *first* when they only have time to review a
handful of pages this week. Today that choice is largely manual or based on a single flat CTR threshold that
ignores position. This lane's output is a ranked queue instead: "review these pages first, and here's why."

**Who acts, and what they do:** an FlyRank content/SEO reviewer (or their client) opens the top of the ranked
queue and rewrites the title tag, meta description, or snippet structure for the highest-ranked candidates —
a cheap, reversible content edit, not a rebuild.

**Cost of a wrong call:** the cost here is mostly *wasted reviewer time*, not damage — if the model points a
reviewer at a page whose CTR gap turns out to be noise (low volume, a mismatched intent, a temporary SERP
feature), they spend 10–20 minutes rewriting a title that didn't need it. That is a real but bounded cost,
which is why precision@K (of the top K candidates, how many are genuine gaps) is the metric that matches the
decision — not recall, and not overall accuracy. A missed genuine gap (false negative) just means it waits
another week for the next queue; it is not a page that "breaks."

**Why data/ML helps at all:** a flat "CTR < X%" rule silently penalizes every page ranked below position 5,
since CTR mechanically falls with position — you'd flag almost the entire long tail and miss real page-1
under-performers. The right comparison (a page vs. its own position tier, adjusted for intent/content type)
has too many interacting factors to hand-write as a single if-statement, but it's exactly the kind of tangled,
multi-signal pattern a simple model or a residual/gap calculation can capture cleanly and explain.


In [4]:
# How big is the decision surface? (supports "unit of analysis" and "who acts" above)
visible = df[(df.avg_position > 0) & (df.impressions_90d >= 500)]
print(f"pages with real visibility (avg_position>0, impressions_90d>=500): {len(visible):,} "
      f"of {len(df):,} total ({len(visible)/len(df)*100:.1f}%)")
print(f"these span {visible.client_id.nunique()} of {df.client_id.nunique()} clients")


pages with real visibility (avg_position>0, impressions_90d>=500): 16,726 of 30,000 total (55.8%)
these span 28 of 32 clients


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [5]:
# 2-3 real numbers that justify this lane

# 1) CTR is not comparable across position tiers -- confirms position-adjustment is necessary
tier_ctr = (df[(df.avg_position > 0) & (df.impressions_90d >= 100)]
            .groupby("position_tier")["ctr"].agg(["mean", "median", "count"])
            .sort_values("mean"))
print("Mean/median CTR (%) by position tier, impressions_90d>=100:")
print(tier_ctr, "\n")

# 2) Size of the review-candidate pool using the starter pipeline's own reason-code rule
#    (impressions_90d>=500, 0<avg_position<=20, ctr<0.5) -- shows the lane has enough volume to matter
low_ctr = df[(df.impressions_90d >= 500) & (df.avg_position > 0) & (df.avg_position <= 20) & (df.ctr < 0.5)]
visible_pool = df[(df.impressions_90d >= 500) & (df.avg_position > 0) & (df.avg_position <= 20)]
print(f"low_ctr_visible_page candidates: {len(low_ctr):,} rows "
      f"({len(low_ctr)/len(df)*100:.1f}% of all 30,000)")
print(f"share of the visible high-impression pool that these candidates make up: "
      f"{len(low_ctr)/len(visible_pool)*100:.1f}% of {len(visible_pool):,} visible pages\n")

# 3) Within-tier CTR gap -- shows "under-performs its own tier" is a real, non-trivial signal
sub = df[(df.avg_position > 0) & (df.impressions_90d >= 500)].copy()
sub["tier_median_ctr"] = sub.groupby("position_tier")["ctr"].transform("median")
below_tier_median = sub[sub["ctr"] < sub["tier_median_ctr"]]
print(f"of {len(sub):,} visible high-impression pages, "
      f"{len(below_tier_median):,} ({len(below_tier_median)/len(sub)*100:.1f}%) sit below their "
      f"own position tier's median CTR -- these are the candidates a flat threshold would miss or mis-rank.")


Mean/median CTR (%) by position tier, impressions_90d>=100:
                   mean  median  count
position_tier                         
deep           0.055415    0.00    879
page_3_5       0.142359    0.06   6058
striking       0.255782    0.15   5903
top_3          0.334128    0.19    533
page_1         0.354760    0.23   8633 

low_ctr_visible_page candidates: 9,759 rows (32.5% of all 30,000)
share of the visible high-impression pool that these candidates make up: 81.2% of 12,023 visible pages

of 16,726 visible high-impression pages, 7,950 (47.5%) sit below their own position tier's median CTR -- these are the candidates a flat threshold would miss or mis-rank.


## 4. Careful words: what I can and can't claim

**What this work CAN claim, if it holds up:**

- *Observed* associations: which pages, on this snapshot, sit below their position tier's typical CTR,
  and by how much.
- A *decision-support* ranking: an ordered review queue with reason codes a human can inspect and override.
- *Directional* language throughout the write-up ("this page's CTR is lower than similar-position peers,"
  "this suggests a title/meta review is worth a look") — never a guarantee of outcome.

**What this work will NOT claim:**

- That editing a title or meta description *caused* a later CTR increase — that requires a real experiment
  (e.g., a before/after test on the pages actually edited), which this observational data cannot provide.
- That a low relative CTR reveals anything about Google's ranking algorithm, AI citations, or AI search
  visibility — this data only measures what happened (clicks vs. impressions), not why.
- That "low CTR" always means a bad title — consolidation, seasonality, and plain low-volume noise can all
  produce the same pattern (see the lane guide's decline-vs-consolidation-vs-noise section), so the final
  queue needs minimum-volume filters and a by-hand look at a sample of top/middle/edge candidates before
  anyone trusts the reason codes.
- Any client name, URL, domain, query, or title from the underlying data — everything here is a
  pseudonymized, aggregated number.


In [6]:
# Sanity check: how much of the "low CTR" signal could plausibly be noise from thin volume?
# (motivates the minimum-volume filter promised in section 4)
thin = df[(df.impressions_90d >= 500) & (df.impressions_90d < 1000)]
thick = df[df.impressions_90d >= 1000]
print(f"candidates with 500-999 impressions (thinner evidence): {len(thin):,}")
print(f"candidates with 1000+ impressions (sturdier evidence): {len(thick):,}")


candidates with 500-999 impressions (thinner evidence): 3,214
candidates with 1000+ impressions (sturdier evidence): 13,512
